À partir des fichiers STAN_DATA_TFP_essential & DATA_STAN on map, aggrège et calcul les différents ratios pour nos estimations.
Ce code retourne un fichier (panel_va_STAN.csv) en panel long, que l'on merge avec nos métriques dans le code création_panel_F.
Un second fichier (intensite_propre.csv) donne l'intensité de R&D propre RDV / PRODV par région, secteur et année (cf. la section en bas du notebook).

In [17]:
import pandas as pd
import numpy as np
import importlib
import project_settings
importlib.reload(project_settings)
from project_settings import PATH_WRK

In [18]:
panel = pd.read_csv(PATH_WRK + 'code/création_panel_estimation/DATA/STAN_DATA_TFP_essential (euro courant).csv')
prod = pd.read_csv(PATH_WRK + 'code/création_panel_estimation/DATA/DATA_STAN (euro courant).csv')

C:\Users\oscar\AppData\Local\Temp\ipykernel_32708\3044499932.py:2: DtypeWarning: Columns (0: EST_METHOD, 1: Méthode d'estimation) have mixed types. Specify dtype option on import or set low_memory=False.
  prod = pd.read_csv(PATH_WRK + 'code/création_panel_estimation/DATA/DATA_STAN (euro courant).csv')


In [19]:
prod.columns

Index(['STRUCTURE', 'STRUCTURE_ID', 'STRUCTURE_NAME', 'ACTION', 'FREQ',
       'Fréquence d'observation', 'REF_AREA', 'Zone de référence', 'ACTIVITY',
       'Activité économique', 'MEASURE', 'Mesure', 'PRICE_BASE',
       'Type de prix', 'UNIT_MEASURE', 'Unité de mesure', 'TIME_PERIOD',
       'Période temporelle', 'OBS_VALUE', 'Valeur d'observation', 'BASE_PER',
       'Période de base', 'UNIT_MULT', 'Multiplicateur d'unité', 'DECIMALS',
       'Décimales', 'EST_METHOD', 'Méthode d'estimation', 'OBS_STATUS',
       'Statut d'observation'],
      dtype='str')

In [20]:
prod['Mesure'].unique()

<ArrowStringArray>
[                     'Valeur ajoutée', 'Valeur ajoutée au coût des facteurs',
                          'Production',           'Rémunération des salariés',
     'Formation brute de capital fixe',                'Stock de capital net']
Length: 6, dtype: str

In [21]:
panel = panel.rename(columns={'TIME_PERIOD' : 'annee',
                              'REF_AREA' : 'Region',
                              'ACTIVITY' : 'Sector'})

prod = prod.rename(columns={'TIME_PERIOD' : 'annee',
                              'REF_AREA' : 'Region',
                              'ACTIVITY' : 'Sector'})

On tri les valeurs négatives et on les remplace en NaN

In [22]:
panel['OBS_VALUE'] = np.where(panel['OBS_VALUE'] < 0, np.nan, panel['OBS_VALUE'])

Mapping commun avec le mapping final de nos bases

In [23]:
df_sector = pd.read_excel(PATH_WRK + "mapping/final_aggregation/dict_oecd_stan.xlsx", sheet_name="Sector", dtype=str)

dict_oecd = {
    code: (None if pd.isna(sector) else sector)
    for code, sector in zip(df_sector["oecd_code"], df_sector["Sector"])
}

df_region = pd.read_excel(PATH_WRK + "mapping/final_aggregation/dict_region_stan.xlsx", sheet_name="region", dtype=str)

dict_region = {
    code: (None if pd.isna(region) else region)
    for code, region in zip(df_region["oecd_code"], df_region["Region"])
}

prod['Activity_STAN'] = prod['Sector']   # STAN code before the mapping (used for the intensity coverage check)

panel['Sector'] = panel['Sector'].map(dict_oecd)
panel["Region"] = panel["Region"].map(dict_region)

prod['Sector'] = prod['Sector'].map(dict_oecd)
prod["Region"] = prod["Region"].map(dict_region)

Création de panel pour chaque mesure

In [24]:
VA = panel[panel['Measure'] == 'Value added'].copy()
VA_costFactors = panel[panel['Measure'] == 'Value added at factor costs'].copy()
comp_emp = panel[panel['Measure'] == 'Compensation of employees'].copy()
emp = panel[panel['Measure'] == 'Employees'].copy()
net_stock_capital = panel[panel['Measure'] == 'Net capital stock'].copy()
production = prod[(prod['Mesure'] == 'Production') & (prod['PRICE_BASE'] == 'V')].copy()   # 'V' = current prices ('L' = chain-linked volume, would double the sum)

In [25]:
VA_V = VA[VA['Price base'] == 'Current prices']
VA_Q = VA[VA['Price base'] == 'Chain linked volume']

In [26]:
VA_V = VA_V.drop(columns=['Measure', 'Price base', 'Unit of measure', 'Unit multiplier'])
VA_Q = VA_Q.drop(columns=['Measure', 'Price base', 'Unit of measure', 'Unit multiplier'])
VA_costFactors = VA_costFactors.drop(columns=['Measure', 'Price base', 'Unit of measure', 'Unit multiplier'])
comp_emp = comp_emp.drop(columns=['Measure', 'Price base', 'Unit of measure', 'Unit multiplier'])
emp = emp.drop(columns=['Measure', 'Price base', 'Unit of measure', 'Unit multiplier'])
net_stock_capital = net_stock_capital.drop(columns=['Measure', 'Price base', 'Unit of measure', 'Unit multiplier'])
production = production.drop(columns=['STRUCTURE', 'STRUCTURE_ID', 'STRUCTURE_NAME', 'ACTION', 'FREQ',
                                      "Fréquence d'observation", "Zone de référence", "Activité économique", 'Mesure', 'PRICE_BASE', 'Activity_STAN',
                                      'Type de prix', 'UNIT_MEASURE', 'Unité de mesure', 'Période temporelle',
                                      "Valeur d'observation", 'BASE_PER', 'Période de base', 'UNIT_MULT', "Multiplicateur d'unité",
                                      'DECIMALS', "Décimales", "EST_METHOD", "Méthode d'estimation", 'OBS_STATUS', "Statut d'observation"])

Aggrégation après le mapping

In [27]:
VA_V = VA_V.groupby(["Region", "Sector", "annee"], as_index=False)["OBS_VALUE"].sum().reset_index()
VA_Q = VA_Q.groupby(["Region", "Sector", "annee"], as_index=False)["OBS_VALUE"].sum().reset_index()
VA_costFactors = VA_costFactors.groupby(["Region", "Sector", "annee"], as_index=False)["OBS_VALUE"].sum().reset_index()
comp_emp = comp_emp.groupby(["Region", "Sector", "annee"], as_index=False)["OBS_VALUE"].sum().reset_index()
emp = emp.groupby(["Region", "Sector", "annee"], as_index=False)["OBS_VALUE"].sum().reset_index()
net_stock_capital = net_stock_capital.groupby(["Region", "Sector", "annee"], as_index=False)["OBS_VALUE"].sum().reset_index()
production = production.groupby(["Region", "Sector", "annee"], as_index=False)["OBS_VALUE"].sum().reset_index()

On renomme les colonnes de chaque panel pour les merger tous ensemble

In [28]:
VA_V = VA_V.rename(columns={"OBS_VALUE": "VA_V"})
VA_Q = VA_Q.rename(columns={"OBS_VALUE": "VA_Q"})
VA_costFactors = VA_costFactors.rename(columns={"OBS_VALUE": "VA_costFactors"})
comp_emp = comp_emp.rename(columns={"OBS_VALUE": "comp_emp"})
emp = emp.rename(columns={"OBS_VALUE": "emp"})
net_stock_capital = net_stock_capital.rename(columns={"OBS_VALUE": "net_stock_capital"})
production = production.rename(columns={"OBS_VALUE": "production"})

In [29]:
pan = VA_V[['Region', 'Sector', 'annee', 'VA_V']].merge(VA_Q[['Region', 'Sector', 'annee', 'VA_Q']], on=['Region', 'Sector', 'annee'], how='inner')
pan = pan.merge(VA_costFactors[['Region', 'Sector', 'annee', 'VA_costFactors']], on=['Region', 'Sector', 'annee'], how='inner')
pan = pan.merge(comp_emp[['Region', 'Sector', 'annee', 'comp_emp']], on=['Region', 'Sector', 'annee'], how='inner')
pan = pan.merge(emp[['Region', 'Sector', 'annee', 'emp']], on=['Region', 'Sector', 'annee'], how='inner')
pan = pan.merge(net_stock_capital[['Region', 'Sector', 'annee', 'net_stock_capital']], on=['Region', 'Sector', 'annee'], how='inner')
pan = pan.merge(production[['Region', 'Sector', 'annee', 'production']], on=['Region', 'Sector', 'annee'], how='inner')

Calculs des différents ratios & TFP avec formule de Cobb-Douglas à rendements d'échelle nuls

In [30]:
pan['VA_V'] = np.where(pan['VA_V'] == 0, np.nan, pan['VA_V'])

pan['1_alpha'] = (pan['comp_emp'] / pan['VA_V'])
pan['alpha'] = 1 - pan['1_alpha']

pan["R_K"] = pan['VA_V'] - pan['comp_emp']
pan['C_k'] = pan['R_K'] / pan['net_stock_capital']
pan['w_l'] = pan['comp_emp'] / pan['emp']
pan['c_u'] = pan['VA_V'] / pan['VA_Q']

pan['produit'] = pan['net_stock_capital']**pan['alpha'] * pan['emp']**pan['1_alpha']
pan['produit'] = np.where(pan['produit'] == 0, np.nan, pan['produit'])

pan['TFP_2'] = pan['VA_Q'] / pan['produit']

pan['Y_L'] = pan['VA_Q'] / pan['emp']
pan['K_L'] = pan['net_stock_capital'] / pan['emp']

In [31]:
pan.to_csv('Panels_intermediaires/panel_va_STAN.csv')

In [32]:
pan.to_csv('Panels_intermediaires/panel_cit1.csv')

## Intensité de R&D propre : RDV / PRODV

Intensité = flux de R&D de l'année **en euros courants** (`RDV (c,s).csv`) / production STAN **aux prix courants en euros** (même année, même secteur, même pays).

- Le calcul ne dépend **pas** de la base MRIO ni du choix du stock (SRDV / SRDQ) : il est fait une seule fois ici, et `création_panel_F` ne fait que lire le fichier `Panels_intermediaires/intensite_propre.csv`.
- Numérateur : `RDV (c,s)` agrégé sur les secteurs communs (`agg_rd_to_*` puis `map_agg_*`, identiques pour toutes les bases) et sur les 23 régions de `dict_region`.
- Dénominateur : production STAN, lignes `PRICE_BASE = 'V'` uniquement, converties en euros au taux de change **de l'année** (le fichier `(euro courant)` est converti au taux fixe 2020 : on le corrige avec `taux 2020 / taux de l'année`). Les pays de la zone euro ont un taux de 1.
- Secteurs agrégés C et H : le numérateur est la somme des codes NACE `C*` / `H*`, le dénominateur est le code STAN natif C / H (même périmètre : H = H49 à H53, sans J).
- Une intensité est laissée vide (NaN) si la production est ≤ `MIN_PROD` M€, si le taux de change manque, ou si un secteur composite n'a pas tous ses codes STAN (ex. `H53+J`) : un dénominateur partiel gonflerait l'intensité.


In [33]:
RD_FILE = PATH_WRK + "RDV (c,s).csv"                       # flux de R&D en valeur, M€ courants (séparateur ",")
MAP_RD_DIR = PATH_WRK + "mapping/agg_rd_to_base/"       # agg_rd_to_<BASE>.xlsx : nace_code -> secteur de la base
MAP_AGG_DIR = PATH_WRK + "mapping/final_aggregation/"   # map_agg_<BASE>.xlsx : secteur de la base -> secteur commun
DATA_DIR = PATH_WRK + "code/création_panel_estimation/DATA/"
MAP_BASE = "EXIOBASE"   # les mappings nace -> secteur commun sont identiques pour les 7 bases (vérifié) : on en prend un
MIN_PROD = 1.0          # M€ : en dessous, l'intensité est considérée comme non définie
REF_YEAR = 2020         # année de référence du taux de change fixe utilisé dans les fichiers (euro courant)


In [34]:
# ---- taux de change annuels (unités de monnaie nationale pour 1 euro) -----------------------------------------------------
pays = pd.read_excel(DATA_DIR + "pays_annees_STAN.xlsx", sheet_name="Pays")
cur_of_region = {dict_region[c]: cur for c, cur in zip(pays["Code"], pays["currency_cd"]) if dict_region.get(c)}

r = pd.read_excel(DATA_DIR + "Exchange_rates_euro_STAN.xlsx", sheet_name="Sheet 1", header=None)
hdr = r.iloc[8]
year_cols = [(j, int(float(hdr[j]))) for j in range(2, r.shape[1])
             if pd.notna(hdr[j]) and str(hdr[j]).replace(".", "").isdigit()]
rate = {}
for i in range(10, r.shape[0]):
    code = r.iloc[i, 0]
    if pd.isna(code) or code in (":", "Special value"):
        continue
    for j, y in year_cols:
        try:
            rate[(code, y)] = float(r.iloc[i, j])
        except (TypeError, ValueError):
            rate[(code, y)] = np.nan


def fx_ratio(region, year):
    """taux 2020 / taux de l'année : passe d'un montant converti au taux fixe 2020 à un montant converti au taux de l'année"""
    cur = cur_of_region.get(region)
    if cur == "EUR":
        return 1.0
    return rate.get((cur, REF_YEAR), np.nan) / rate.get((cur, int(year)), np.nan)


In [35]:
# ---- dénominateur : production STAN aux prix courants, en M€ au taux de change de l'année -------------------------------
regions = sorted(set(v for v in dict_region.values() if v))

p = prod[(prod["Mesure"] == "Production") & (prod["PRICE_BASE"] == "V")].dropna(subset=["Sector", "Region"]).copy()
p["OBS_VALUE"] = np.where(p["OBS_VALUE"] < 0, np.nan, p["OBS_VALUE"])   # valeurs négatives -> NaN (comme pour le panel STAN) ; un 0 reste un 0 (ex. code U)
pairs_ry = p[["Region", "annee"]].drop_duplicates()
pairs_ry["fx"] = [fx_ratio(rg, y) for rg, y in zip(pairs_ry["Region"], pairs_ry["annee"])]
p = p.merge(pairs_ry, on=["Region", "annee"], how="left")
print("lignes sans taux de change :", int(p["fx"].isna().sum()))
p["prod_eur"] = p["OBS_VALUE"] * p["fx"]

# un secteur composite n'est valide que si tous ses codes STAN sont présents
n_expected = pd.Series(dict_oecd).dropna().value_counts()
den = (p.dropna(subset=["prod_eur"])
        .groupby(["Region", "Sector", "annee"])
        .agg(production_V=("prod_eur", "sum"), n_codes=("Activity_STAN", "nunique"))
        .reset_index())
den["n_expected"] = den["Sector"].map(n_expected)
den["complet"] = den["n_codes"] >= den["n_expected"]
print("cellules (région, secteur, année) incomplètes (mises à NaN) :", int((~den["complet"]).sum()), "sur", len(den))
den.loc[~den["complet"], "production_V"] = np.nan
den = den[["Region", "Sector", "annee", "production_V"]]


lignes sans taux de change : 0
cellules (région, secteur, année) incomplètes (mises à NaN) : 51 sur 23051


In [36]:
# ---- numérateur : flux de R&D en valeur (RDV), agrégé sur les secteurs communs ----------------------------------------------
rd = pd.read_csv(RD_FILE, sep=",")[["year", "country_code", "nace_code", "rd_value_M€"]]
m_sec = pd.read_excel(MAP_RD_DIR + f"agg_rd_to_{MAP_BASE}.xlsx", sheet_name="Sector", dtype=str)
m_reg = pd.read_excel(MAP_RD_DIR + f"agg_rd_to_{MAP_BASE}.xlsx", sheet_name="Region", dtype=str)
m_agg = pd.read_excel(MAP_AGG_DIR + f"map_agg_{MAP_BASE}.xlsx", sheet_name="Sector", dtype=str)

rd["Region"] = rd["country_code"].map(dict(zip(m_reg["country_code"], m_reg["aggregation"])))
rd["Sector"] = rd["nace_code"].map(dict(zip(m_sec["nace_code"], m_sec["aggregation"])))
rd["Sector"] = rd["Sector"].map(dict(zip(m_agg["Sector"], m_agg["aggregation"])))
rd = rd[rd["Region"].isin(regions)].rename(columns={"year": "annee", "rd_value_M€": "rd_propre_V"})
print("codes NACE de R&D sans secteur commun :", sorted(rd.loc[rd["Sector"].isna(), "nace_code"].unique()))

num = rd.groupby(["Region", "Sector", "annee"], as_index=False)["rd_propre_V"].sum(min_count=1)
# agrégats C et H : somme des codes NACE C* et H* (H = H49 à H53, sans J, comme le code STAN H)
for letter in ["C", "H"]:
    sub = rd[rd["nace_code"].str.startswith(letter)]
    agg = sub.groupby(["Region", "annee"], as_index=False)["rd_propre_V"].sum(min_count=1)
    agg["Sector"] = letter
    num = pd.concat([num, agg], ignore_index=True)


codes NACE de R&D sans secteur commun : []


In [37]:
# --- intensité = RDV / PRODV -----------------------------------------------------------------------------------------------
intensite = den.merge(num, on=["Region", "Sector", "annee"], how="left")
intensite["intensite_V"] = np.where(intensite["production_V"] > MIN_PROD,
                                    intensite["rd_propre_V"] / intensite["production_V"], np.nan)
intensite = intensite[["Region", "Sector", "annee", "rd_propre_V", "production_V", "intensite_V"]]
intensite.to_csv(PATH_WRK + "code/création_panel_estimation/Panels_intermediaires/intensite_propre.csv", index=False)

print(intensite.shape, "| régions :", intensite["Region"].nunique(), "| secteurs :", intensite["Sector"].nunique(),
      "| années :", intensite["annee"].min(), "-", intensite["annee"].max())
print("intensité définie :", int(intensite["intensite_V"].notna().sum()), "sur", len(intensite))
intensite.groupby("Sector")["intensite_V"].median().mul(100).round(2).rename("médiane de l'intensité (%)")


(23051, 6) | régions : 23 | secteurs : 36 | années : 1995 - 2022
intensité définie : 22956 sur 23051


Sector
A          0.04
B          0.16
C          1.27
C10_C12    0.20
C13_C15    0.36
C16_C18    0.16
C19        0.16
C20+C21    2.15
C22        0.61
C23        0.34
C24        0.34
C25        0.44
C26        4.72
C27        1.67
C28        1.80
C29        1.11
C30        2.10
C31_C33    0.58
D          0.06
E          0.07
F          0.03
G          0.15
H          0.02
H49        0.01
H50        0.00
H51        0.00
H52        0.03
H53+J      1.07
I          0.00
K          0.12
L          0.00
M+N        1.19
O+U        0.00
P          0.00
Q          0.01
R+S+T      0.02
Name: médiane de l'intensité (%), dtype: float64